# 06 · Texto: codificar con pocos ejemplos (semi-supervisado)
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrezconz/investigacion-aumentada-ml/blob/main/notebooks/06_texto_semisupervisado.ipynb)

**Situación:** tienes cientos de fragmentos de relatorías, entrevistas o respuestas abiertas, y codificarlos todos a mano no es viable.

**Ruta:** (1) escribes un libro de códigos en .md, (2) codificas a mano unos pocos fragmentos, (3) el modelo propone el código del resto, (4) revisas una muestra.

**Los 32 fragmentos de este ejercicio son ficticios**, escritos para la práctica con el tono de una relatoría comunitaria rural.

In [ ]:
# Preparación: trae los datos del repositorio (funciona igual en Google Colab)
import os, urllib.request
REPO = "https://raw.githubusercontent.com/andrezconz/investigacion-aumentada-ml/main/datos/"

def traer(nombre):
    local = os.path.join("..", "datos", nombre)
    if os.path.exists(local):
        return local
    if not os.path.exists(nombre):
        urllib.request.urlretrieve(REPO + nombre, nombre)
    return nombre

In [ ]:
import pandas as pd
df = pd.read_csv(traer("fragmentos_practica.csv"))
print(df["codigo_investigador"].value_counts().to_dict())
df.head(8)

## Paso 2 · Codificamos a mano solo 2 fragmentos por categoría
El resto queda **sin etiqueta** (−1). La columna `codigo_investigador` la guardamos aparte solo para revisar al final.

In [ ]:
codificados_a_mano = [1, 2, 9, 10, 17, 18, 25, 26]
codigos = sorted(df["codigo_investigador"].unique())
a_num = {c: i for i, c in enumerate(codigos)}

y = df["codigo_investigador"].map(a_num).where(df["id"].isin(codificados_a_mano), -1)
print("Con etiqueta:", int((y >= 0).sum()), "· sin etiqueta:", int((y < 0).sum()))

## Paso 3 · El modelo propone el resto
**Cómo funciona:** convierte cada fragmento en números según sus palabras (TF-IDF), aprende de los 8 codificados, etiqueta los casos de los que está más seguro y vuelve a aprender incluyendo esos. Repite hasta no poder avanzar.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier

vacias = ["el", "la", "los", "las", "de", "del", "y", "a", "en", "que", "se", "no", "un", "una", "por", "para",
          "con", "al", "es", "nos", "lo", "hay", "cada", "desde", "porque", "muy", "ni", "su", "sus"]
tfidf = TfidfVectorizer(strip_accents="unicode", lowercase=True, stop_words=vacias, ngram_range=(1, 2), sublinear_tf=True)
X = tfidf.fit_transform(df["fragmento"])

modelo = SelfTrainingClassifier(LogisticRegression(C=10, max_iter=2000), threshold=0.30, max_iter=20).fit(X, y)
df["propuesta"] = [codigos[i] for i in modelo.predict(X)]
df["prob"] = modelo.predict_proba(X).max(axis=1).round(2)

## Paso 4 · Revisar
Comparamos la propuesta con la codificación del investigador, **solo en los fragmentos que el modelo no vio etiquetados**.

In [ ]:
nuevos = ~df["id"].isin(codificados_a_mano)
acierto = (df.loc[nuevos, "propuesta"] == df.loc[nuevos, "codigo_investigador"]).mean()
print(f"Coincidencia con el investigador: {acierto * 100:.0f} % de {nuevos.sum()} fragmentos")
print("\nDesacuerdos (revisar a mano):")
df.loc[nuevos & (df["propuesta"] != df["codigo_investigador"]), ["fragmento", "codigo_investigador", "propuesta", "prob"]]

**¿Por qué acierta tan poco?** Con 32 textos cortos y solo 8 codificados, contar palabras no alcanza: fragmentos del mismo tema casi no comparten palabras («acueducto», «quebrada», «tanque»). Es el límite de **pocos datos** que vimos en el bloque 3. Hay dos salidas: codificar más ejemplos a mano o usar un modelo que entienda el significado, como un asistente de IA con tu libro de códigos (abajo).

> **Tip:** la validez la pone tu revisión, no el modelo. Si los desacuerdos se repiten en un mismo código, el problema suele estar en el libro de códigos: ajusta definiciones y ejemplos, y repite.

## La misma ruta con un asistente de IA
Con un modelo de lenguaje no hace falta entrenar nada: le das el libro de códigos y él propone. Esta celda arma el texto para pegar en Claude, ChatGPT o Gemini.

In [ ]:
try:
    libro = open(os.path.join("..", "ficha", "libro_de_codigos.md")).read()
except FileNotFoundError:
    import urllib.request
    libro = urllib.request.urlopen(REPO.replace("/datos/", "/ficha/") + "libro_de_codigos.md").read().decode()

muestra = df.sample(5, random_state=1)
instruccion = libro + "\n\n## Fragmentos por codificar\n" + "\n".join(f"{r.id}. {r.fragmento}" for r in muestra.itertuples())
instruccion += "\n\nDevuelve una tabla con: id, código asignado y una frase que justifique la decisión con palabras del fragmento."
print(instruccion)